# Kodra AI Agent: Cloud/Colab GPU Training Preparation Notebook

**Product:** Kodra AI Agent  
**Model:** Kodra GPT (`KodraGPT`)  
**Core:** Kodra Core  
**Tagline:** CODE • THINK • CREATE

This notebook prepares and validates a GPU training run for **Kodra GPT** on Google Colab (or any Jupyter environment with a CUDA GPU). It clones the repository, prepares an approved dataset, trains or loads a BPE tokenizer, selects a Kodra Tiny/Small configuration with its exact parameter count, runs a 20-step GPU smoke test with train/validation loss reporting, saves and reloads a checkpoint, resumes for 5 more steps, and optionally backs checkpoints up to Google Drive.

Full multi-epoch training is behind an explicit **RUN MANUALLY ONLY** gate near the end of the notebook — it will not run as part of "Run All".

No credentials are embedded in this notebook. If you want to persist checkpoints to Google Drive, mount it yourself in Colab and pass that path as `CHECKPOINT_DIR` below.

In [ ]:
# 1. Clone the repository and enter the Kodra Core directory
!git clone git@github.com:ChaceEthan/Kodra-ai.git
%cd Kodra-ai/kodra-core

In [ ]:
# 2. Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# 3. Detect GPU and print info
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device name:', torch.cuda.get_device_name(0))
    print('Device count:', torch.cuda.device_count())
    print('Total memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print('No GPU detected - training will fall back to CPU (slow for anything above kodra-tiny).')

In [ ]:
# 4. (Optional) Mount Google Drive for persistent checkpoint storage.
# Uncomment if you want checkpoints to survive a Colab session restart.
# from google.colab import drive
# drive.mount('/content/drive')
# CHECKPOINT_DIR = '/content/drive/MyDrive/kodra_checkpoints'
CHECKPOINT_DIR = 'checkpoints'

In [ ]:
# 5. Prepare dataset: build a manifest from an approved/licensed source tree.
# Point SOURCE_DIR at a directory you have the rights to train on. The bundled
# `data/code` sample corpus is used by default so this cell always runs.
from datasets.corpus_pipeline import build_manifest, write_manifest

SOURCE_DIR = 'data/code'
manifest = build_manifest(SOURCE_DIR, seed=42, license='project-sample', source='kodra-sample-corpus')
write_manifest(manifest, 'data/manifest.json')
print(f'Discovered {manifest.num_files} files, {manifest.total_chars} chars, languages: {manifest.language_counts}')

In [ ]:
# 6. Train (or load) the tokenizer. BPE is the default for GPU runs; the
# Phase 1 char tokenizer remains available for parity/debug comparisons.
import os
from datasets.sample_code import SAMPLE_CODE_CORPUS
from tokenizer.char_tokenizer import CharTokenizer
from tokenizer.bpe_tokenizer import ByteLevelBPETokenizer

USE_BPE = True  # set False to use the Phase 1 char tokenizer instead
BPE_VOCAB_PATH = 'tokenizer/vocab_bpe.json'
CHAR_VOCAB_PATH = 'tokenizer/vocab.json'

if USE_BPE:
    tokenizer = ByteLevelBPETokenizer(vocab_size=8000)
    if os.path.exists(BPE_VOCAB_PATH):
        tokenizer.load(BPE_VOCAB_PATH)
        print(f'Loaded existing BPE tokenizer from {BPE_VOCAB_PATH}')
    else:
        tokenizer.train(SAMPLE_CODE_CORPUS)
        tokenizer.save(BPE_VOCAB_PATH)
        print(f'Trained a new BPE tokenizer and saved it to {BPE_VOCAB_PATH}')
else:
    tokenizer = CharTokenizer()
    if os.path.exists(CHAR_VOCAB_PATH):
        tokenizer.load(CHAR_VOCAB_PATH)
        print(f'Loaded existing char tokenizer from {CHAR_VOCAB_PATH}')
    else:
        tokenizer.train(SAMPLE_CODE_CORPUS)
        tokenizer.save(CHAR_VOCAB_PATH)
        print(f'Trained a new char tokenizer and saved it to {CHAR_VOCAB_PATH}')

print('Tokenizer type:', tokenizer.tokenizer_type, '| vocab size:', tokenizer.vocab_size)

In [ ]:
# 7. Select a Kodra model configuration
from configs.model_sizes import get_model_size, validate_model_config, estimate_resources
from model.gpt_model import KodraGPT

MODEL_SIZE = 'kodra-tiny'  # one of: kodra-tiny, kodra-small (kodra-base/kodra-medium are roadmap-only)
spec = get_model_size(MODEL_SIZE)
model_cfg = spec.config
model_cfg.vocab_size = tokenizer.vocab_size
validate_model_config(model_cfg)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = KodraGPT(model_cfg).to(device)

# Exact parameter count comes from the instantiated model, not the roadmap estimate.
param_count = model.count_parameters()
resource_estimate = estimate_resources(model_cfg)
print(f'{spec.display_name}: {param_count:,} exact parameters ({param_count/1e6:.2f}M) on {device}')
print(f'Previously trained in this repo: {spec.trained}')
print(f'Rough planning estimate: {resource_estimate["parameters"]:,} params | '
      f'training_vram~{resource_estimate["training_vram_gb"]:.2f}GB | '
      f'inference_vram~{resource_estimate["inference_vram_gb"]:.2f}GB')

In [ ]:
# 8. Build train/validation dataloaders and the trainer
from configs.config import TrainingConfig
from datasets.dataset import create_dataloader
from training.trainer import Trainer
from training.utils import set_seed

set_seed(42)

# Simple held-out split of the sample corpus for a train/validation loss signal.
_split_idx = int(len(SAMPLE_CODE_CORPUS) * 0.9)
TRAIN_TEXT = SAMPLE_CODE_CORPUS[:_split_idx]
VAL_TEXT = SAMPLE_CODE_CORPUS[_split_idx:]

train_cfg = TrainingConfig(batch_size=16, learning_rate=3e-4, max_epochs=10)
train_loader = create_dataloader(TRAIN_TEXT, tokenizer, model_cfg.context_length, train_cfg.batch_size)
val_loader = create_dataloader(VAL_TEXT, tokenizer, model_cfg.context_length, train_cfg.batch_size, shuffle=False)

dataset_manifest_id = f"{manifest.source}-seed{manifest.seed}-{manifest.created_at}"

trainer = Trainer(
    model, train_cfg, train_loader, val_loader=val_loader, device=device,
    tokenizer_type=tokenizer.tokenizer_type, dataset_manifest_id=dataset_manifest_id,
)

In [ ]:
# 9. GPU smoke training: run a fixed 20 optimizer steps to validate the
# pipeline end-to-end (forward/backward/step, AMP, checkpointing) before
# committing to a full run. This is NOT the full training loop.
SMOKE_TRAIN_STEPS = 20

smoke_epoch = 0
while trainer.step_count < SMOKE_TRAIN_STEPS:
    smoke_epoch += 1
    smoke_avg_loss = trainer.train_epoch(smoke_epoch, total_steps=SMOKE_TRAIN_STEPS)
    print(f'[smoke] epoch {smoke_epoch} | avg_loss={smoke_avg_loss:.4f} | step={trainer.step_count}')

print(f'Smoke training complete at step {trainer.step_count} (target was {SMOKE_TRAIN_STEPS}).')

In [ ]:
# 15. FULL TRAINING — RUN MANUALLY ONLY
#
# The smoke test above already validated the pipeline. Full training is a
# long-running, resource-consuming operation and must be started deliberately.
# Flip RUN_FULL_TRAINING to True yourself to proceed; it defaults to False so
# re-running the whole notebook (e.g. "Run All") never triggers a full run.
RUN_FULL_TRAINING = False

if not RUN_FULL_TRAINING:
    raise RuntimeError(
        'Full training is gated. Set RUN_FULL_TRAINING = True above and re-run this '
        'cell to start the full training loop.'
    )

# Resume from the latest checkpoint if one exists (continues past the smoke/resume steps above).
full_ckpt = os.path.join(CHECKPOINT_DIR, 'kodra_gpt_latest.pt')
if os.path.exists(full_ckpt):
    trainer.load_checkpoint(full_ckpt)
    print(f'Resumed full training from step {trainer.step_count}')
else:
    print('No existing checkpoint found - starting full training fresh.')

total_steps = train_cfg.max_epochs * len(train_loader)
for epoch in range(1, train_cfg.max_epochs + 1):
    avg_loss = trainer.train_epoch(epoch, total_steps=total_steps)
    val_loss = trainer.evaluate()
    trainer.save_latest_and_best(CHECKPOINT_DIR, val_loss=val_loss)
    tps = trainer.history[-1]['tokens_per_sec'] if trainer.history else 0.0
    print(f'Epoch {epoch}/{train_cfg.max_epochs} | avg_loss={avg_loss:.4f} | '
          f'val_loss={val_loss} | step={trainer.step_count} | tokens/sec={tps:.0f}')

In [ ]:
# 16. Evaluate (language-model + code-completion + syntax)
from evaluation.evaluator import full_evaluation_report
import json as _json

report = full_evaluation_report(model, tokenizer, device, train_loader)
print(_json.dumps(report, indent=2, default=str))

In [ ]:
# 17. Generate a sample code completion with the trained model
from inference.generator import CodeGenerator

generator = CodeGenerator(model, tokenizer, device)
sample = generator.generate('def quicksort(arr):', max_new_tokens=64, temperature=0.7, top_k=40)
print(sample)

In [ ]:
# 13. 5-step resume: continue training the reloaded trainer for 5 more steps
RESUME_STEPS = 5
resume_target = reload_trainer.step_count + RESUME_STEPS
resume_epoch = 0
while reload_trainer.step_count < resume_target:
    resume_epoch += 1
    resume_avg_loss = reload_trainer.train_epoch(resume_epoch, total_steps=resume_target)
    print(f'[resume] epoch {resume_epoch} | avg_loss={resume_avg_loss:.4f} | step={reload_trainer.step_count}')

print(f'Resumed training to step {reload_trainer.step_count} (target was {resume_target}).')

In [ ]:
# 14. (Optional) Back up checkpoints to Google Drive. Only runs if Drive is
# mounted (see the optional cell above) and BACKUP_TO_DRIVE is set True.
BACKUP_TO_DRIVE = False
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/kodra_checkpoints_backup'

if BACKUP_TO_DRIVE:
    import shutil
    if not os.path.isdir('/content/drive'):
        print('Google Drive is not mounted - skipping backup. Mount it in the cell above first.')
    else:
        os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
        for fname in os.listdir(CHECKPOINT_DIR):
            shutil.copy2(os.path.join(CHECKPOINT_DIR, fname), os.path.join(DRIVE_BACKUP_DIR, fname))
        print(f'Backed up checkpoints from {CHECKPOINT_DIR} to {DRIVE_BACKUP_DIR}')
else:
    print('BACKUP_TO_DRIVE is False - skipping Drive backup.')

In [ ]:
# 10. Train, checkpointing periodically (every epoch here; adjust to your needs)
total_steps = train_cfg.max_epochs * len(train_loader)
for epoch in range(1, train_cfg.max_epochs + 1):
    avg_loss = trainer.train_epoch(epoch, total_steps=total_steps)
    val_loss = trainer.evaluate()  # None unless a val_loader was configured
    trainer.save_latest_and_best(CHECKPOINT_DIR, val_loss=val_loss)
    tps = trainer.history[-1]['tokens_per_sec'] if trainer.history else 0.0
    print(f'Epoch {epoch}/{train_cfg.max_epochs} | avg_loss={avg_loss:.4f} | step={trainer.step_count} | tokens/sec={tps:.0f}')

In [ ]:
# 11. Evaluate (language-model + code-completion + syntax)
from evaluation.evaluator import full_evaluation_report
import json as _json

report = full_evaluation_report(model, tokenizer, device, train_loader)
print(_json.dumps(report, indent=2, default=str))

In [ ]:
# 12. Generate a sample code completion with the trained model
from inference.generator import CodeGenerator

generator = CodeGenerator(model, tokenizer, device)
sample = generator.generate('def quicksort(arr):', max_new_tokens=64, temperature=0.7, top_k=40)
print(sample)